In [2]:
from transformers import pipeline
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax


In [3]:
import google.protobuf
print("protobuf installed successfully")


protobuf installed successfully


In [4]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    truncation=True,
    max_length=512
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)


In [6]:
MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)

model = AutoModelForSequenceClassification.from_pretrained(MODEL)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
text = "Good night 😊"
text = preprocess(text)

print("Processed text:", text)


Processed text: Good night 😊


In [8]:
encoded_input = tokenizer(text, return_tensors='pt')
encoded_input


{'input_ids': tensor([[    0, 18621, 17431,     6, 82803,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [9]:
output = model(**encoded_input)


In [10]:
scores = output[0][0].detach().numpy()
scores = softmax(scores)

scores


array([0.03125937, 0.20148005, 0.7672607 ], dtype=float32)

In [ ]:
ranking = np.argsort(scores)[::-1]

for i in range(scores.shape[0]):
    label = config.id2label[ranking[i]]
    score = scores[ranking[i]]
    print(f"{i+1}) {label} {np.round(float(score), 4)}") 


1) positive 0.7673
2) neutral 0.2015
3) negative 0.0313
